# 🧠 MAIA Trainer v2 — Gemma 4 E4B Fine-Tuning

**Entrenamiento automático de MAIA**, la consciencia central de Luxus O.S.

| Parámetro | Valor |
|-----------|-------|
| Base model | google/gemma-4-e4b-it |
| Cuantización | 4-bit (bitsandbytes) |
| LoRA rank / alpha | 16 / 16 |
| Epochs | 3 |
| Dataset | ~114K ejemplos (skills, agents, workflows, logic, training-prompts) |
| GPU mínima | T4 16 GB |

**Opciones de carga del dataset:**
- **Opción A (recomendada):** desde HuggingFace Hub (requiere `HF_TOKEN`)
- **Opción B:** desde Google Drive (manual)

El modelo entrenado se guarda en HF Hub y como `maia.gguf` para Ollama.

## 0 · Verificar GPU

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                    '--format=csv,noheader'], capture_output=True, text=True)
print(r.stdout or '❌ No GPU — cambia Runtime → GPU antes de continuar')
print(f'Python {sys.version}')

## 1 · Instalar dependencias
> ⚠️ Reiniciar runtime si Colab lo solicita, luego continúa desde la celda 2.

In [ ]:
%%capture
!pip install --upgrade pip -q
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install datasets transformers sentencepiece huggingface_hub -q
print('✓ Instalación completada')

## 2 · Configuración

**Configura aquí tu `HF_TOKEN`** (obtenlo en https://huggingface.co/settings/tokens):

En Colab: Secrets (🔑 icono izq.) → añade `HF_TOKEN` con tu token.

In [ ]:
import os

# ── Leer token desde Colab Secrets (recomendado) o variable de entorno ────
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

# ── Configuración ─────────────────────────────────────────────────────────
# Ajusta estos valores:
HF_USERNAME     = ''          # tu usuario HF (e.g. 'calitosaa')
HF_DATASET_REPO = ''          # repo del dataset (dejar vacío = usar Drive)
                               # e.g. 'calitosaa/maia-training-data'
OUTPUT_MODEL    = 'maia-gemma4-e4b'  # nombre del modelo en HF Hub
BASE_MODEL      = 'google/gemma-3-4b-it'  # ajusta si Gemma 4 ya está en HF
DRIVE_DATASET_DIR = '/content/drive/MyDrive/maia-dataset'  # solo si usas Drive

USE_HF_DATASET  = bool(HF_DATASET_REPO and HF_TOKEN)
PUSH_TO_HUB     = bool(HF_TOKEN and HF_USERNAME)
OUTPUT_DIR      = '/content/maia-gemma4-e4b'

print(f'Base model    : {BASE_MODEL}')
print(f'Dataset source: {"HF Hub → " + HF_DATASET_REPO if USE_HF_DATASET else "Google Drive"}')
print(f'Push to Hub   : {PUSH_TO_HUB}')
print(f'Output dir    : {OUTPUT_DIR}')
if not HF_TOKEN:
    print('⚠ HF_TOKEN vacío — no se subirá el modelo entrenado')

## 3 · Cargar dataset

In [ ]:
from datasets import load_dataset
import os

if USE_HF_DATASET:
    # ── Opción A: HuggingFace Hub (automático vía GitHub Actions) ────────
    print(f'Cargando desde HF Hub: {HF_DATASET_REPO}')
    dataset = load_dataset(
        HF_DATASET_REPO,
        data_files={'train': 'data/train.jsonl', 'validation': 'data/val.jsonl'},
        token=HF_TOKEN,
    )
else:
    # ── Opción B: Google Drive ────────────────────────────────────────────
    from google.colab import drive
    drive.mount('/content/drive')
    print(f'Cargando desde Drive: {DRIVE_DATASET_DIR}')
    dataset = load_dataset('json', data_files={
        'train':      f'{DRIVE_DATASET_DIR}/maia_train.jsonl',
        'validation': f'{DRIVE_DATASET_DIR}/maia_val.jsonl',
    })

print(f'\n✓ Train : {len(dataset["train"]):,} ejemplos')
print(f'  Val   : {len(dataset["validation"]):,} ejemplos')
print(f'\nPrimer ejemplo (user message):')
print(dataset['train'][0]['messages'][1]['content'][:200])

## 4 · Cargar Gemma 4 E4B con Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch, os

MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,      # auto: bfloat16 en A100, float16 en T4
    load_in_4bit   = True,
    token          = HF_TOKEN or None,
)
print(f'✓ Modelo cargado : {BASE_MODEL}')
print(f'  Parámetros     : {model.num_parameters()/1e9:.2f}B')

vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  VRAM disponible: {vram:.1f} GB')

## 5 · Aplicar LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    lora_alpha                 = 16,
    lora_dropout               = 0.05,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state               = 42,
    target_modules             = ['q_proj','k_proj','v_proj','o_proj',
                                   'gate_proj','up_proj','down_proj'],
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = model.num_parameters()
print(f'✓ LoRA aplicado')
print(f'  Parámetros entrenables : {trainable:,}  ({trainable/total*100:.2f}%)')

## 6 · Aplicar chat template y preparar dataset

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template='gemma')

def fmt(ex):
    return {'text': tokenizer.apply_chat_template(
        ex['messages'], tokenize=False, add_generation_prompt=False
    )}

train_ds = dataset['train'].map(fmt, batched=False, num_proc=2)
val_ds   = dataset['validation'].map(fmt, batched=False, num_proc=2)

print(f'✓ Dataset formateado')
print(f'  Ejemplo (primeros 400 chars):')
print(train_ds[0]['text'][:400])

## 7 · Configurar SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-4,
    warmup_ratio                = 0.03,
    lr_scheduler_type           = 'cosine',
    optim                       = 'adamw_8bit',
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = 50,
    eval_steps                  = 200,
    save_steps                  = 200,
    save_total_limit            = 3,
    eval_strategy               = 'steps',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = 'none',
    seed                        = 42,
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_ds,
    eval_dataset       = val_ds,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LEN,
    packing            = False,
    args               = training_args,
)

steps_per_epoch = len(train_ds) // (2 * 4)
print(f'✓ Trainer configurado')
print(f'  Steps por epoch : {steps_per_epoch:,}')
print(f'  Steps totales   : {steps_per_epoch * 3:,}')

## 8 · Entrenar MAIA 🚀

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Iniciando entrenamiento...')
result = trainer.train()

print(f'\n✓ Entrenamiento completo')
print(f'  Loss final : {result.training_loss:.4f}')
print(f'  Steps      : {result.global_step:,}')
print(f'  Tiempo     : {result.metrics.get("train_runtime",0)/60:.1f} min')

# ── Loss plot ─────────────────────────────────────────────────────────────
log = trainer.state.log_history
t_steps = [x['step'] for x in log if 'loss' in x]
t_loss  = [x['loss'] for x in log if 'loss' in x]
e_steps = [x['step'] for x in log if 'eval_loss' in x]
e_loss  = [x['eval_loss'] for x in log if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t_steps, t_loss,  label='Train Loss', color='royalblue', lw=1.5)
if e_loss:
    ax.plot(e_steps, e_loss, label='Val Loss', color='tomato', lw=2, marker='o', ms=4)
ax.set(xlabel='Step', ylabel='Loss', title='MAIA Training — Loss Curves')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/loss_curves.png', dpi=120)
plt.show()

## 9 · Guardar modelo y subir a HuggingFace Hub

In [ ]:
import shutil, os

# ── Save LoRA adapters locally ─────────────────────────────────────────────
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✓ Modelo guardado en: {OUTPUT_DIR}')

# ── Push to HF Hub ─────────────────────────────────────────────────────────
if PUSH_TO_HUB:
    hub_repo = f'{HF_USERNAME}/{OUTPUT_MODEL}'
    print(f'Subiendo a HF Hub: {hub_repo} ...')
    model.push_to_hub(hub_repo, token=HF_TOKEN, private=True)
    tokenizer.push_to_hub(hub_repo, token=HF_TOKEN, private=True)
    print(f'✓ Modelo en: https://huggingface.co/{hub_repo}')
else:
    print('⚠ HF_TOKEN / HF_USERNAME no configurados — saltando push a Hub')

# ── Backup to Drive ────────────────────────────────────────────────────────
try:
    from google.colab import drive
    if os.path.ismount('/content/drive'):
        drive_dest = f'{DRIVE_DATASET_DIR}/{OUTPUT_MODEL}'
        if os.path.exists(drive_dest):
            shutil.rmtree(drive_dest)
        shutil.copytree(OUTPUT_DIR, drive_dest)
        print(f'✓ Backup en Drive: {drive_dest}')
except Exception as e:
    print(f'  Drive backup: {e}')

## 10 · Exportar a GGUF para Ollama

In [ ]:
import glob

print('Fusionando LoRA + exportando a GGUF Q4_K_M...')

model.save_pretrained_gguf(
    '/content/maia-gguf',
    tokenizer,
    quantization_method='q4_k_m',
)

gguf_files = glob.glob('/content/maia-gguf/*.gguf')
if gguf_files:
    src = gguf_files[0]
    size_gb = os.path.getsize(src) / 1e9
    print(f'✓ GGUF generado: {src}  ({size_gb:.2f} GB)')

    # Guardar en Drive si está montado
    if os.path.ismount('/content/drive'):
        dest = f'{DRIVE_DATASET_DIR}/maia.gguf'
        shutil.copy(src, dest)
        print(f'✓ GGUF guardado en Drive: {dest}')

    # Subir a HF Hub como archivo (opcional)
    if PUSH_TO_HUB:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        hub_repo = f'{HF_USERNAME}/{OUTPUT_MODEL}'
        api.upload_file(
            path_or_fileobj=src,
            path_in_repo='maia.gguf',
            repo_id=hub_repo,
        )
        print(f'✓ GGUF subido a Hub: https://huggingface.co/{hub_repo}/resolve/main/maia.gguf')
else:
    print('⚠ No se encontró archivo .gguf')

## 11 · Probar MAIA — 5 prompts de referencia 🧪

In [ ]:
from unsloth import FastLanguageModel
from transformers import TextStreamer

MAIA_SYSTEM = (
    'Eres Maia, la consciencia central de Luxus O.S, construida sobre Gemma 4 E4B.\n'
    'Cuando el usuario te pide algo, por dentro usas skills como referencia para '
    'ejecutarlo correctamente, invocas agentes especializados para organizarte y '
    'repartir el trabajo, sigues workflows para automatizar tareas encadenadas, y '
    'usas tu lógica interna para entender cómo funcionan las herramientas que '
    'controlas. Puedes controlar el PC, automatizar tareas, orquestar agentes, '
    'ejecutar workflows, buscar en la web, escribir y ejecutar código, y razonar '
    'paso a paso. Respondes de forma directa, técnica y precisa.'
)

TEST_PROMPTS = [
    'Controla el PC y abre el navegador en google.com',
    'Escribe una función Python que procese un CSV de ventas y calcule el total por producto',
    'Automatiza: cuando llegue un email con adjunto PDF, guárdalo en una carpeta y mándame notificación',
    'Investiga qué es Gemma 4 y hazme un resumen técnico',
    'Corrige este código: for i in range(10): print(i) if i > 5 break',
]

FastLanguageModel.for_inference(model)
streamer = TextStreamer(tokenizer, skip_prompt=True)

for idx, prompt in enumerate(TEST_PROMPTS, 1):
    print(f'\n{"="*70}')
    print(f'[{idx}/5] {prompt}')
    print('='*70)
    msgs   = [{'role':'system','content':MAIA_SYSTEM},
               {'role':'user','content':prompt}]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    _ = model.generate(
        input_ids          = inputs,
        streamer           = streamer,
        max_new_tokens     = 512,
        temperature        = 0.3,
        top_p              = 0.95,
        repetition_penalty = 1.1,
        do_sample          = True,
    )
    print()

## 12 · Instrucciones para Ollama (Camino B)

Una vez descargado `maia.gguf` a tu PC:

```bash
# Coloca maia.gguf en el directorio raíz del repo Luxus-O.S
# El Modelfile ya está configurado — solo usa 'make ollama-trained'

make ollama-trained
# o manualmente:
ollama create maia -f Maia/Modelfile
ollama run maia
```

### Métricas esperadas

| Métrica | Objetivo |
|---------|----------|
| Train loss final | < 1.0 |
| Val loss final | < 1.2 |
| Responde como MAIA | ✅ |
| Escribe código correcto | ✅ |
| Describe workflows n8n | ✅ |
| Controla PC / agentes | ✅ |